# Lab: Quản Trị Dữ Liệu & Bảo Mật

**Khóa học:** AICB-P2T2 · Lab #24  
**Thời gian:** 3–4 giờ  
**Hình thức:** Cá nhân hoặc nhóm 2 người

---

## Bối cảnh

Bạn vừa gia nhập team AI của **MedViet** — một startup y tế Việt Nam xử lý hồ sơ bệnh nhân để huấn luyện mô hình chẩn đoán. Công ty đang chuẩn bị ký hợp đồng doanh nghiệp và cần đạt chuẩn **NĐ13/2023** và **ISO 27001**.

Nhiệm vụ của bạn: xây dựng **pipeline quản trị dữ liệu** end-to-end, sau đó bổ sung **quản trị agent** để các AI agent không thể vượt qua các lớp kiểm soát.

### Những gì bạn sẽ xây dựng

| Trụ cột | Công nghệ | Phần lab |
|--------|-----------|----------|
| Phát hiện & ẩn danh PII | Microsoft Presidio + spaCy | Phần 2 |
| Kiểm soát truy cập theo vai trò | Casbin RBAC | Phần 3 |
| Mã hóa dữ liệu lưu trữ | AES-256-GCM envelope encryption | Phần 4 |
| Chất lượng dữ liệu | Great Expectations | Phần 5 |
| Quản trị agent | [Agent Governance Toolkit](https://github.com/microsoft/agent-governance-toolkit) | Phần 6 |
| Ánh xạ tuân thủ | Checklist NĐ13 | Phần 7 |

### Cấu trúc project

```
Day24-Track02-Lab-Assignments/
├── medviet-governance/          # Project code chính (hoàn thành TODO tại đây)
├── agent-governance-toolkit/    # SDK quản trị agent của Microsoft (tham khảo)
└── data-governance-lab/         # Notebook này
```

> **Gợi ý:** Mở `medviet-governance/` ở tab khác. Hầu hết TODO nằm trong `src/`; notebook này hướng dẫn từng phần một cách tương tác.

## Phần 0 — Cài đặt môi trường

Chạy cell này một lần khi bắt đầu lab.

In [1]:
import os
import sys
from pathlib import Path

# Xác định thư mục gốc của project
LAB_ROOT = Path.cwd()
if LAB_ROOT.name != "data-governance-lab":
    LAB_ROOT = LAB_ROOT / "data-governance-lab"

REPO_ROOT = LAB_ROOT.parent
MEDVIET_ROOT = REPO_ROOT / "medviet-governance"
AGT_ROOT = REPO_ROOT / "agent-governance-toolkit"

sys.path.insert(0, str(MEDVIET_ROOT))
sys.path.insert(0, str(LAB_ROOT))

print(f"Thư mục lab:       {LAB_ROOT}")
print(f"Project MedViet:   {MEDVIET_ROOT}  (tồn tại={MEDVIET_ROOT.exists()})")
print(f"Bộ công cụ AGT:    {AGT_ROOT}  (tồn tại={AGT_ROOT.exists()})")

Thư mục lab:       /mnt/c/Users/quand/OneDrive/Documents/VinUni-AI2k_2/Day24-Track02-Lab/data-governance-lab
Project MedViet:   /mnt/c/Users/quand/OneDrive/Documents/VinUni-AI2k_2/Day24-Track02-Lab/medviet-governance  (tồn tại=True)
Bộ công cụ AGT:    /mnt/c/Users/quand/OneDrive/Documents/VinUni-AI2k_2/Day24-Track02-Lab/agent-governance-toolkit  (tồn tại=False)


In [2]:
# === CÀI ĐẶT (bỏ comment từng dòng khi cần) ===

# 1. Dependencies cơ bản
# !pip install -r requirements.txt

# 2. Model tiếng Việt (KHÔNG dùng `spacy download`)
!pip install --break-system-packages pyvi
!pip install --break-system-packages --no-deps https://gitlab.com/trungtv/vi_spacy/-/raw/master/packages/vi_core_news_lg-3.6.0/dist/vi_core_news_lg-3.6.0.tar.gz

# 3. Agent Governance Toolkit — Phần 6
# Cách A: từ PyPI
!pip install --break-system-packages "agent-governance-toolkit[full]"
# Cách B: từ source local (repo đã có trong project)
!pip install --break-system-packages -e "../agent-governance-toolkit/agent-governance-python/agent-governance-toolkit-core[full]"

# 4. Kiểm tra AGT đã cài đúng
!python -c "from agent_os import StatelessKernel; from agent_os.policies import PolicyEvaluator; print('✓ AGT OK')"

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached vi_core_news_lg-3.6.0.tar.gz (233.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for vi_core_news_lg: filename=vi_core_news_lg-3.6.0-py3-none-any.whl size=233275698 sha256=9b0d2aef30e8ed561e0d04cedd9a2a268e318f4a700bd69a9cff30190d6a1007
  Stored in directory: /home/quandum/.cache/pip/wheels/0c/cc/a3/bbdb17097672910ac0ba537631fbd782522b23752d485c117c
Successfully built vi_core_news_lg
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
ERROR: ../agent-governance-toolkit/agent-governance-python/agent-governance-toolkit-core[full] is not a valid editable requirement. It should either be a path to a local projec

---
## Phần 1 — Chuẩn bị dữ liệu (15 phút)

Tạo dữ liệu bệnh nhân giả lập tiếng Việt chứa PII thực tế.

**Bài 1.1:** Chạy generator và kiểm tra kết quả. Liệt kê tất cả các cột chứa PII.

In [3]:
import pandas as pd
from faker import Faker
import random

fake = Faker("vi_VN")
Faker.seed(42)
random.seed(42)

def generate_patients(n: int = 200) -> pd.DataFrame:
    records = []
    for _ in range(n):
        records.append({
            "patient_id": fake.uuid4(),
            "ho_ten": fake.name(),
            "cccd": str(random.randint(0, 9)) + "".join(str(random.randint(0, 9)) for _ in range(11)),
            "ngay_sinh": fake.date_of_birth(minimum_age=18, maximum_age=90).strftime("%d/%m/%Y"),
            "so_dien_thoai": f"0{random.choice([3, 5, 7, 8, 9])}" + "".join(str(random.randint(0, 9)) for _ in range(8)),
            "email": fake.email(),
            "dia_chi": fake.address(),
            "benh": random.choice(["Tiểu đường", "Huyết áp cao", "Tim mạch", "Khỏe mạnh"]),
            "ket_qua_xet_nghiem": round(random.uniform(3.5, 12.0), 2),
            "bac_si_phu_trach": fake.name(),
            "ngay_kham": fake.date_this_year().strftime("%d/%m/%Y"),
        })
    return pd.DataFrame(records)

raw_dir = MEDVIET_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
raw_path = raw_dir / "patients_raw.csv"

df_raw = generate_patients(200)
df_raw.to_csv(raw_path, index=False)
print(f"Đã tạo {len(df_raw)} bản ghi → {raw_path}")
df_raw.head(3)

Đã tạo 200 bản ghi → /mnt/c/Users/quand/OneDrive/Documents/VinUni-AI2k_2/Day24-Track02-Lab/medviet-governance/data/raw/patients_raw.csv


,patient_id,ho_ten,cccd,ngay_sinh,so_dien_thoai,email,dia_chi,benh,ket_qua_xet_nghiem,bac_si_phu_trach,ngay_kham
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Quang Phạm,104332181960,05/09/1945,0313389083,maijohn@example.net,"338 John Số\nThị xã JaneThành phố, 895667",Khỏe mạnh,5.37,Quý cô Thành Vũ,28/01/2026
1,1a2a73ed-562b-4f79-8374-59eef50bea63,Kim Trần,940265423511,12/09/1961,0815594078,jane07@example.net,"849 Hoàng Làng\nThành phố JohnHuyện, 183667",Tiểu đường,11.77,Cô Hồng Vũ,23/03/2026
2,5ec42e08-29a3-42e9-9d65-a441d58842de,Trung Đức Trần,618495931034,17/12/1979,0331647525,johnle@example.com,"5 John Đường\nJaneHuyện, 169403",Tim mạch,5.28,Bà Mai Dương,13/03/2026


In [ ]:
# BÀI 1.1 — Liệt kê các cột PII
PII_COLUMNS = [
    "ho_ten",           # Họ tên bệnh nhân
    "cccd",             # Căn cước công dân (12 số)
    "ngay_sinh",        # Ngày sinh
    "so_dien_thoai",    # Số điện thoại
    "email",            # Địa chỉ email
    "dia_chi",          # Địa chỉ nhà
    "bac_si_phu_trach", # Bác sĩ phụ trách
]

NON_PII_COLUMNS = [c for c in df_raw.columns if c not in PII_COLUMNS]
print("Cột PII:", PII_COLUMNS)
print("Cột không phải PII (an toàn cho ML):", NON_PII_COLUMNS)

---
## Phần 2 — Phát hiện & ẩn danh PII (45 phút)

Dùng **Microsoft Presidio** với các recognizer tùy chỉnh cho CCCD và số điện thoại Việt Nam.

Hoàn thành các TODO trong `medviet-governance/src/pii/detector.py` và `anonymizer.py`, sau đó chạy các cell bên dưới để kiểm tra.

In [ ]:
from pii_utils import build_vietnamese_analyzer, detect_pii, print_model_setup_hint

analyzer = build_vietnamese_analyzer(cache_dir=LAB_ROOT)
print_model_setup_hint()
print("Analyzer đã sẵn sàng.")

In [ ]:
# Kiểm tra nhanh phát hiện PII
test_cases = [
    "Bệnh nhân Nguyen Van A, CCCD: 012345678901",
    "Liên hệ: 0912345678",
    "Email: nguyenvana@gmail.com",
]

for text in test_cases:
    results = detect_pii(text, analyzer)
    entities = [(r.entity_type, text[r.start:r.end]) for r in results]
    print(f"{text!r}")
    print(f"  → {entities}\n")

In [ ]:
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

def anonymize_sample(text: str, strategy: str = "mask") -> str:
    """Ẩn danh tối giản cho demo notebook. Phiên bản đầy đủ: MedVietAnonymizer trong src/pii/anonymizer.py"""
    results = detect_pii(text, analyzer)
    if not results:
        return text
    operators = {
        "PERSON": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 50, "from_end": False}),
        "EMAIL_ADDRESS": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 50, "from_end": False}),
        "VN_CCCD": OperatorConfig("replace", {"new_value": "************"}),
        "VN_PHONE": OperatorConfig("replace", {"new_value": "0*********"}),
    }
    engine = AnonymizerEngine()
    return engine.anonymize(text=text, analyzer_results=results, operators=operators).text

sample = "Nguyen Van A, CCCD 012345678901, SĐT 0912345678, email a@b.com"
print("Trước:", sample)
print("Sau: ", anonymize_sample(sample))

In [ ]:
# BÀI 2.4 — Tính tỷ lệ phát hiện (mục tiêu: ≥ 95%)
pii_columns = ["ho_ten", "cccd", "so_dien_thoai", "email"]
sample_df = df_raw.head(50)

total, detected = 0, 0
for col in pii_columns:
    for value in sample_df[col].astype(str):
        total += 1
        if len(detect_pii(value, analyzer)) > 0:
            detected += 1

detection_rate = detected / total if total else 0.0
print(f"Tỷ lệ phát hiện: {detection_rate:.1%} ({detected}/{total})")
assert detection_rate >= 0.95, f"Dưới ngưỡng 95% — cải thiện recognizer trong detector.py"

---
## Phần 3 — RBAC với Casbin (30 phút)

Định nghĩa ai được truy cập tài nguyên dữ liệu nào. Hoàn thành `medviet-governance/src/access/policy.csv` và `rbac.py`.

| Vai trò | Đọc PII thô | Đọc dữ liệu huấn luyện | Xóa dữ liệu |
|---------|-------------|------------------------|-------------|
| admin | ✅ | ✅ | ✅ |
| ml_engineer | ❌ | ✅ | ❌ |
| data_analyst | ❌ | ❌ (chỉ aggregated) | ❌ |
| intern | ❌ | ❌ (chỉ sandbox) | ❌ |

In [ ]:
import casbin

model_path = MEDVIET_ROOT / "src" / "access" / "model.conf"
policy_path = MEDVIET_ROOT / "src" / "access" / "policy.csv"

enforcer = casbin.Enforcer(str(model_path), str(policy_path))

def check_access(role: str, resource: str, action: str) -> bool:
    return enforcer.enforce(role, resource, action)

# BÀI 3.1 — Kiểm tra quyền truy cập sau khi hoàn thành policy.csv
tests = [
    ("admin", "patient_data", "read", True),
    ("ml_engineer", "patient_data", "read", False),
    ("ml_engineer", "training_data", "read", True),
    ("data_analyst", "aggregated_metrics", "read", True),
    ("intern", "sandbox_data", "read", True),
    ("intern", "patient_data", "read", False),
]

for role, resource, action, expected in tests:
    result = check_access(role, resource, action)
    status = "✓" if result == expected else "✗"
    print(f"{status} {role} → {action} {resource}: {result} (mong đợi {expected})")

---
## Phần 4 — Mã hóa envelope (30 phút)

Triển khai **mã hóa envelope AES-256-GCM** trong `medviet-governance/src/encryption/vault.py`.

```
Master Key (KEK) → mã hóa → Data Key (DEK) → mã hóa → Dữ liệu bệnh nhân
```

> Trong production, KEK phải lưu trong HSM hoặc cloud KMS — không bao giờ lưu trong file thường.

In [ ]:
import os
import base64
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

class SimpleVault:
    """Triển khai tham khảo cho lab notebook."""

    def __init__(self, master_key_path: str = ".vault_key_demo"):
        self.master_key_path = master_key_path
        self.kek = self._load_or_create_kek()

    def _load_or_create_kek(self) -> bytes:
        if os.path.exists(self.master_key_path):
            return base64.b64decode(open(self.master_key_path, "rb").read())
        kek = os.urandom(32)
        with open(self.master_key_path, "wb") as f:
            f.write(base64.b64encode(kek))
        return kek

    def generate_dek(self) -> tuple[bytes, bytes]:
        plaintext_dek = os.urandom(32)
        aesgcm = AESGCM(self.kek)
        nonce = os.urandom(12)
        encrypted_dek = nonce + aesgcm.encrypt(nonce, plaintext_dek, None)
        return plaintext_dek, encrypted_dek

    def decrypt_dek(self, encrypted_dek: bytes) -> bytes:
        nonce, ciphertext = encrypted_dek[:12], encrypted_dek[12:]
        return AESGCM(self.kek).decrypt(nonce, ciphertext, None)

    def encrypt_data(self, plaintext: str) -> dict:
        plaintext_dek, encrypted_dek = self.generate_dek()
        aesgcm = AESGCM(plaintext_dek)
        nonce = os.urandom(12)
        ciphertext = nonce + aesgcm.encrypt(nonce, plaintext.encode(), None)
        del plaintext_dek
        return {
            "encrypted_dek": base64.b64encode(encrypted_dek).decode(),
            "ciphertext": base64.b64encode(ciphertext).decode(),
            "algorithm": "AES-256-GCM",
        }

    def decrypt_data(self, payload: dict) -> str:
        encrypted_dek = base64.b64decode(payload["encrypted_dek"])
        blob = base64.b64decode(payload["ciphertext"])
        plaintext_dek = self.decrypt_dek(encrypted_dek)
        nonce, ciphertext = blob[:12], blob[12:]
        plaintext = AESGCM(plaintext_dek).decrypt(nonce, ciphertext, None)
        del plaintext_dek
        return plaintext.decode()

vault = SimpleVault(master_key_path=str(LAB_ROOT / ".vault_key_demo"))
original = "Nguyen Van A - CCCD: 012345678901"
encrypted = vault.encrypt_data(original)
decrypted = vault.decrypt_data(encrypted)

print("Gốc:     ", original)
print("Đã mã hóa:", {k: v[:40] + "..." if len(v) > 40 else v for k, v in encrypted.items()})
print("Giải mã: ", decrypted)
assert decrypted == original, "Round-trip thất bại!"
print("\n✓ Mã hóa envelope round-trip thành công")

---
## Phần 5 — Kiểm tra chất lượng dữ liệu (20 phút)

Dùng **Great Expectations** để áp đặt ràng buộc schema và giá trị lên dữ liệu bệnh nhân.

Hoàn thành `medviet-governance/src/quality/validation.py`.

In [ ]:
# Kiểm tra nhẹ không cần thiết lập GX context đầy đủ
def validate_patient_data(df: pd.DataFrame) -> dict:
    failed = []

    if df["patient_id"].isnull().any():
        failed.append("patient_id có giá trị null")
    if not (df["cccd"].astype(str).str.len() == 12).all():
        failed.append("cccd phải đúng 12 chữ số")
    if not df["ket_qua_xet_nghiem"].between(0, 50).all():
        failed.append("ket_qua_xet_nghiem ngoài khoảng [0, 50]")
    valid_benh = {"Tiểu đường", "Huyết áp cao", "Tim mạch", "Khỏe mạnh"}
    if not df["benh"].isin(valid_benh).all():
        failed.append("benh chứa giá trị không hợp lệ")
    if df["patient_id"].duplicated().any():
        failed.append("phát hiện patient_id trùng lặp")

    return {"success": len(failed) == 0, "failed_checks": failed, "total_rows": len(df)}

result = validate_patient_data(df_raw)
print(f"Kiểm tra: {'ĐẠT' if result['success'] else 'KHÔNG ĐẠT'}")
if result["failed_checks"]:
    for check in result["failed_checks"]:
        print(f"  ✗ {check}")

---
## Phần 6 — Quản trị Agent với Microsoft AGT (30 phút)

Quản trị dữ liệu kiểm soát **ai** được truy cập dữ liệu. **Quản trị agent** kiểm soát **AI agent được phép làm gì về mặt cấu trúc** — kể cả khi prompt injection cố vượt qua quy tắc.

[Agent Governance Toolkit](https://github.com/microsoft/agent-governance-toolkit) chặn mọi lệnh gọi tool *trước khi* thực thi. Hành động bị từ chối là **không thể thực hiện**, không phải "khó xảy ra".

### Cài đặt AGT (bắt buộc trước khi chạy cell bên dưới)

**Yêu cầu:** Python 3.11+

**Cách A — từ PyPI** (cần internet):
```bash
pip install "agent-governance-toolkit[full]"
```

**Cách B — từ source local** (repo `agent-governance-toolkit/` đã có trong project, khuyến nghị):
```bash
cd data-governance-lab
pip install -e "../agent-governance-toolkit/agent-governance-python/agent-governance-toolkit-core[full]"
```

**Kiểm tra:**
```bash
python -c "from agent_os import StatelessKernel; from agent_os.policies import PolicyEvaluator; print('AGT OK')"
```

> Sau khi cài, **restart kernel** Jupyter rồi chạy lại Phần 0.

### Khái niệm chính

1. **PolicyEvaluator** — quy tắc YAML/lập trình (allow / deny / require_approval)
2. **StatelessKernel** — sandbox thực thi có kiểm tra policy
3. **AgentIdentity** — định danh mật mã cho từng agent (audit trail)

File policy: `policies/medviet-data-policy.yaml`

In [ ]:
# Cài AGT (bỏ comment một trong hai cách), rồi restart kernel
# !pip install "agent-governance-toolkit[full]"
# !pip install -e "../agent-governance-toolkit/agent-governance-python/agent-governance-toolkit-core[full]"

try:
    from agent_os import StatelessKernel
    from agent_os.policies import PolicyEvaluator
    print("✓ AGT đã cài — chạy cell tiếp theo")
except ImportError:
    print("⚠ Chưa cài AGT. Chạy pip ở trên, restart kernel, chạy lại Phần 0.")

In [ ]:
import asyncio
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*agent-os-kernel.*")

try:
    from agent_os import StatelessKernel, ExecutionContext
    from agent_os.policies import (
        PolicyEvaluator, PolicyDocument, PolicyRule,
        PolicyCondition, PolicyAction, PolicyOperator, PolicyDefaults,
    )
    AGT_AVAILABLE = True
except ImportError:
    AGT_AVAILABLE = False
    print("⚠ Chưa cài Agent Governance Toolkit. Chọn một cách:")
    print('  pip install "agent-governance-toolkit[full]"')
    print('  pip install -e "../agent-governance-toolkit/agent-governance-python/agent-governance-toolkit-core[full]"')
    print("  Sau đó restart kernel và chạy lại Phần 0.")

if AGT_AVAILABLE:
    medviet_policy = PolicyDocument(
        name="medviet-data-policy",
        version="1.0",
        defaults=PolicyDefaults(action=PolicyAction.DENY),
        rules=[
            PolicyRule(
                name="allow-anonymized-read",
                condition=PolicyCondition(field="dataset", operator=PolicyOperator.EQ, value="anonymized"),
                action=PolicyAction.ALLOW, priority=100,
            ),
            PolicyRule(
                name="block-raw-pii",
                condition=PolicyCondition(field="dataset", operator=PolicyOperator.EQ, value="raw_pii"),
                action=PolicyAction.DENY, priority=200,
            ),
            PolicyRule(
                name="block-export-outside-vn",
                condition=PolicyCondition(field="destination", operator=PolicyOperator.NE, value="VN"),
                action=PolicyAction.DENY, priority=300,
            ),
        ],
    )
    evaluator = PolicyEvaluator(policies=[medviet_policy])
    print("Policy evaluator đã sẵn sàng.")

In [ ]:
if AGT_AVAILABLE:
    agent_actions = [
        {"action": "read", "dataset": "anonymized", "agent_id": "ml-agent-01"},
        {"action": "read", "dataset": "raw_pii", "agent_id": "ml-agent-01"},
        {"action": "export", "dataset": "anonymized", "destination": "US"},
        {"action": "export", "dataset": "anonymized", "destination": "VN"},
    ]

    for ctx in agent_actions:
        result = evaluator.evaluate(ctx)
        allowed = result.allowed if hasattr(result, "allowed") else str(result)
        print(f"{ctx} → {'CHO PHÉP' if allowed else 'TỪ CHỐI'}")

In [ ]:
if AGT_AVAILABLE:
    kernel = StatelessKernel()

    async def governed_data_access(dataset: str, action: str = "read") -> str:
        ctx = ExecutionContext(agent_id="medviet-ml-agent", policies=["read_only"])
        result = await kernel.execute(
            action="data_access",
            params={"dataset": dataset, "action": action},
            context=ctx,
        )
        return result.data if result.success else f"BỊ CHẶN: {result.error}"

    async def _demo():
        print("Đọc có kiểm soát (anonymized):", await governed_data_access("anonymized"))
        print("Thử ghi bị chặn:", await governed_data_access("raw_pii", action="write"))

    await _demo()

### BÀI 6.1 — Thiết kế policy cho agent

Chỉnh sửa `policies/medviet-data-policy.yaml` để thêm quy tắc:
- Cho phép agent `data_analyst` chỉ đọc dataset `aggregated`
- Yêu cầu phê duyệt của con người trước mọi hành động `delete`

Sau đó chạy lại cell đánh giá policy ở trên.

---
## Phần 7 — Ánh xạ tuân thủ (15 phút)

Ánh xạ các biện pháp kỹ thuật với yêu cầu **NĐ13/2023**. Mở `medviet-governance/compliance_checklist.md` và hoàn thành các dòng TODO.

In [ ]:
COMPLIANCE_MAP = {
    "Tối thiểu hóa dữ liệu": {
        "control": "Pipeline ẩn danh PII với Presidio",
        "status": "Hoàn thành",
        "owner": "AI Team",
    },
    "Kiểm soát truy cập": {
        "control": "Casbin RBAC + OPA ABAC + AGT agent policies",
        "status": "Hoàn thành",
        "owner": "Platform Team",
    },
    "Mã hóa": {
        "control": "Mã hóa envelope AES-256-GCM (SimpleVault)",
        "status": "Đang triển khai",
        "owner": "Infra Team",
    },
    "Ghi log kiểm toán": {
        "control": "FastAPI middleware log + CloudTrail cho AWS deployment",
        "status": "Đang triển khai",
        "owner": "Platform Team",
    },
    "Phát hiện vi phạm": {
        "control": "Prometheus + Grafana monitoring + Bandit SAST + git-secrets hook",
        "status": "Đang triển khai",
        "owner": "Security Team",
    },
}

pd.DataFrame(COMPLIANCE_MAP).T

---
## Tổng kết lab & tiêu chí chấm điểm

| Hạng mục | Điểm | Tiêu chí |
|---------|------|---------|
| Phát hiện PII | 25 | Tỷ lệ phát hiện ≥ 95%; CCCD + SĐT + email đều detect được |
| Ẩn danh hóa | 20 | Không còn PII gốc trong output; cột lâm sàng giữ nguyên |
| RBAC | 20 | 4 vai trò hoạt động đúng; trả 403 đúng chỗ |
| Mã hóa | 15 | Round-trip mã hóa envelope thành công |
| Kiểm tra bảo mật | 10 | Hook git-secrets chặn credential; có báo cáo Bandit |
| Tuân thủ | 10 | Ánh xạ NĐ13 đầy đủ với biện pháp kỹ thuật cụ thể |

**Ngưỡng đạt: ≥ 70 / 100**

### Bước tiếp theo

1. Hoàn thành tất cả TODO trong `medviet-governance/src/`
2. Chạy `pytest medviet-governance/tests/ -v`
3. Khởi động API: `uvicorn src.api.main:app --reload` (từ thư mục `medviet-governance/`)
4. Nộp bài: `src/`, `tests/`, `policies/`, `data/processed/`, `compliance_checklist.md`, `reports/`